In [1]:
import pandas as pd
from io import StringIO
import requests
from bs4 import BeautifulSoup
from IPython.core.display import HTML
import re
from collections import Counter
import numpy as np

## Webscraping


In this exercise, you'll practice using BeautifulSoup to parse the content of a web page. The page that you'll be scraping, https://realpython.github.io/fake-jobs/, contains job listings. Your job is to extract the data on each job and convert into a pandas DataFrame.

#### 1. Start by performing a GET request on the url above and convert the response into a BeautifulSoup object.  

In [5]:
## This is the base url from which we're scraping our data
URL = 'https://realpython.github.io/fake-jobs/'

# This is a sort of parameter that can be required to scrape certain sites
headers = {
    "User-Agent": "PretendJobsHunter"
}

# using the requests library get request to a specific url to retrieve data from a web server and includes special headers
response = requests.get(URL, headers=headers)

# using the .text attribute of the response object, requests library attempts to decode the information being passed
# BeautifulSoup completes the job by parsing the HTML contact of the web page
# BeautifulSoup converts raw HTML text into a structured, searchable object htat makes it easy to extract data
mixture = BeautifulSoup(response.text)
#print(mixture.prettify())

#### 1a. Use the .find method to find the tag containing the first job title ("Senior Python Developer"). Hint: can you find a tag type and/or a class that could be helpful for extracting this information? Extract the text from this title.  

In [7]:
# attempt to isolate the line of code I desire
mixture.findAll('h2')[0]#[:5]#.text

<h2 class="title is-5">Senior Python Developer</h2>

In [8]:
# extract only the exact code I desire
mixture.find('h2').text# class_="title").text

'Senior Python Developer'


#### 1b. Now, use what you did for the first title, but extract the job title for all jobs on this page. Store the results in a list.  

In [10]:
#take the soup/mixture and find all h2 tags and save them to job_titles collection/variable
job_titles = mixture.findAll('h2')#[:5]#.text

# for every item in job_titles, use the .text attribute of the response object, requests library attempts to decode the information being passed
list_job_titles = [item.text for item in job_titles]

# inspect the first 2 list items in the collection list_job_titles
list_job_titles[:2]

['Senior Python Developer', 'Energy engineer']

#### 1c. Finally, extract the companies, locations, and posting dates for each job. For example, the first job has a company of "Payne, Roberts and Davis", a location of "Stewartbury, AA", and a posting date of "2021-04-08". Ensure that the text that you extract is clean, meaning no extra spaces or other characters at the beginning or end.  

##### Company names

In [13]:
#take the soup/mixture and find all h3 tags and save them to the company_names variable
company_names = mixture.findAll('h3')#[:5]#.text

# for every item in company_names, use the .text attribute of the response object, requests libruary attempts to decode the info being pased
list_company_names = [i.text for i in company_names]

# inspect the first 2 list items in the collection list_company_names
list_company_names[:2]

['Payne, Roberts and Davis', 'Vasquez-Davidson']

##### Posting Dates

In [15]:
#take the soup/mixture and find all time tags and save them to the posting_dates variable
posting_dates = mixture.findAll('time')

# for every item in posting_dates, use the .text attribute of the response object, requests libruary attempts to decode the info being pased
list_posting_dates = [i.text for i in posting_dates]

# inspect the first 2 list items in the collection list_posting_dates
list_posting_dates[:2]

['2021-04-08', '2021-04-08']

##### Location

In [17]:
#takes the soup/mixture and finds the first p tag and saves it to the locs variable
locs = mixture.find('p', attrs={'class' : 'location'}).text
#checking out what that result looks like
locs

'\n        Stewartbury, AA\n      '

In [18]:
#Attempting to remove unnecessary characters and spaces
location01 = ''.join(locs.split())

# investigating what that result looks like
location01

'Stewartbury,AA'

In [19]:
# use regex to add a space following the comma and capture the entirety of the location
re.sub(r'(?<=[.,])(?=[^\s])', r' ', location01)

'Stewartbury, AA'

In [20]:
# takes the soup/mixture finds all of the p tags with the class labeled location and saves them all to locations
locations = mixture.findAll('p', attrs={'class' : 'location'})#[:2]

# for each line of locations, it uses the .text attribute of hte response object, requests libaray attempts to decode the info being passed
strange_list_locations = [i.text for i in locations]

# inspect result
strange_list_locations[:2]

['\n        Stewartbury, AA\n      ', '\n        Christopherville, AA\n      ']

In [21]:
#Attempting to remove unnecessary characters and spaces for whole list
funky_loc_list = [''.join(i.split()) for i in strange_list_locations]
funky_loc_list[:2]

['Stewartbury,AA', 'Christopherville,AA']

In [22]:
# use regex to add a space following the comma and capture the entirety of the locations in the funky_loc_list
job_locations = [re.sub(r'(?<=[.,])(?=[^\s])', r' ', x) for x in funky_loc_list]

# see what that looks like
job_locations[0]

'Stewartbury, AA'

#### 1d. Take the lists that you have created and combine them into a pandas DataFrame. 

In [24]:
# create a pandas df
pretend_jobs_table = pd.DataFrame(
    # naming the new columns and inputting the list data from the corresponding collections
    {'job_titles': list_job_titles,
     'company': list_company_names,
     'locations': job_locations,
     'posting_dates': list_posting_dates
    })
# calling the top of that df to see what it looks like
pretend_jobs_table.head(2)

,job_titles,company,locations,posting_dates
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08


#### 2a. Next, add a column that contains the url for the "Apply" button. Try this in two ways.  a. First, use the BeautifulSoup find_all method to extract the urls.  

In [26]:
# find the first a tag and inspect it
mixture.find('a')

<a class="card-footer-item" href="https://www.realpython.com" target="_blank">Learn</a>

In [27]:
# find all a tags in the soup/mixture and for every row of that resulting list, use the requests library 
# get request to retrieve every href link and compile into whole_links_list variable
whole_links_list = [link.get('href') for link in mixture.findAll('a')]

# inspect what the first two rows of that looks like
whole_links_list[:8]

['https://www.realpython.com',
 'https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html',
 'https://www.realpython.com',
 'https://realpython.github.io/fake-jobs/jobs/energy-engineer-1.html',
 'https://www.realpython.com',
 'https://realpython.github.io/fake-jobs/jobs/legal-executive-2.html',
 'https://www.realpython.com',
 'https://realpython.github.io/fake-jobs/jobs/fitness-centre-manager-3.html']

In [28]:
# extracts every second element of the whole_links_list and collects them into apply_links
apply_links = whole_links_list[1::2]

In [29]:
# create a pandas df
pretend_jobs_table_links_included = pd.DataFrame(
    # naming the new columns and inputting the list data from the corresponding collections
    {'job_titles': list_job_titles,
     'company': list_company_names,
     'locations': job_locations,
     'posting_dates': list_posting_dates, 
     'apply_link' : apply_links
    })
pretend_jobs_table_links_included['job_titles'][10]

'Software Engineer (Python)'

#### 2b. Next, add a column that contains the url for the "Apply" button. Try this in two ways. b. Next, get those same urls in a different way. Examine the urls and see if you can spot the pattern of how they are constructed. Then, build the url using the elements you have already extracted. Ensure that the urls that you created match those that you extracted using BeautifulSoup. Warning: You will need to do some string cleaning and prep in constructing the urls this way. For example, look carefully at the urls for the "Software Engineer (Python)" job and the "Scientist, research (maths)" job. 

In [31]:
# will use the list of job titles to append to the remaining structure of the apply button link 
#inspect tricky example
list_job_titles[10]

'Software Engineer (Python)'

In [32]:
# remove \ and ( ) from the list of job titles and pull each word into it's own list of strings using regex
removed_slashes_and_parentheses = [re.findall(r"[^()-/]\w+", item) for item in list_job_titles]
#inspect progress
removed_slashes_and_parentheses[70]

['Back', 'End', ' Web', ' Developer', 'Python', ' Django']

In [33]:
#[y if y not in b else other_value for y in a]

In [34]:
# removed the spaces between words and replace them with dashes
add_dashses_between_and_remove_spaces = ['-'.join(item).replace(' ', '') for item in removed_slashes_and_parentheses]
#inspect progress
add_dashses_between_and_remove_spaces[10]

'Software-Engineer-Python'

In [35]:
# add dashes to the end of the titles
needs_nums = [item + "-" for item in add_dashses_between_and_remove_spaces]
#inspect progress
needs_nums[10]

'Software-Engineer-Python-'

In [36]:
# add numbers 0 through 99 to the end of the job title
numbered_job_titles_list = [needs_nums[i] + str(i) for i in range(100)]
# inspect list
numbered_job_titles_list[10]

'Software-Engineer-Python-10'

In [37]:
pretend_jobs_table_links_included['apply_link'][0]

'https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html'

In [38]:
# combine first part of apply button href link add the titles and numbers then end it in .html
frankenstein_apply_link = ["https://realpython.github.io/fake-jobs/jobs/" + item.lower() + ".html" for item in numbered_job_titles_list]

# inspecting first link
frankenstein_apply_link[0]

'https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html'

In [39]:
# create a pandas df
pretend_job_df_both_links_included = pd.DataFrame(
    # naming the new columns and inputting the list data from the corresponding collections
    {'job_titles': list_job_titles,
     'company': list_company_names,
     'locations': job_locations,
     'posting_dates': list_posting_dates, 
     'apply_link' : apply_links,
     'my_apply_link': frankenstein_apply_link
    })

In [40]:
pretend_job_df_both_links_included.head(2)

,job_titles,company,locations,posting_dates,apply_link,my_apply_link
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/se...,https://realpython.github.io/fake-jobs/jobs/se...
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/en...,https://realpython.github.io/fake-jobs/jobs/en...


In [204]:
# check to be sure my links matched the other column of links
pretend_job_df_both_links_included["equal"] = np.where(pretend_job_df_both_links_included["apply_link"] == pretend_job_df_both_links_included["my_apply_link"], True, False)
pretend_job_df_both_links_included["equal"].value_counts()

equal
True    100
Name: count, dtype: int64

In [206]:
#pretend_job_df_both_links_included.groupby("equal")["my_apply_link"].value_counts()

#### 3. Finally, we want to get the job description text for each job.

#### 3a. Start by looking at the page for the first job, https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html. Using BeautifulSoup, extract the job description paragraph.

In [46]:
## This is the base url from which we're scraping our data
URL = 'https://realpython.github.io/fake-jobs/'

# This is a sort of parameter that can be required to scrape certain sites
headers = {
    "User-Agent": "PretendJobsHunter"
}

# using the requests library get request to a specific url to retrieve data from a web server and includes special headers
response = requests.get(URL, headers=headers)

# using the .text attribute of the response object, requests library attempts to decode the information being passed
# BeautifulSoup completes the job by parsing the HTML contact of the web page
# BeautifulSoup converts raw HTML text into a structured, searchable object htat makes it easy to extract data
mixture = BeautifulSoup(response.text)
#print(mixture.prettify())
mixture.findAll('h2')[0]#[:5]#.text

<h2 class="title is-5">Senior Python Developer</h2>

#### 3b. We want to be able to do this for all pages. Write a function which takes as input a url and returns the description text on that page. For example, if you input "https://realpython.github.io/fake-jobs/jobs/television-floor-manager-8.html" into your function, it should return the string "At be than always different American address. Former claim chance prevent why measure too. Almost before some military outside baby interview. Face top individual win suddenly. Parent do ten after those scientist. Medical effort assume teacher wall. Significant his himself clearly very. Expert stop area along individual. Three own bank recognize special good along.".

#### 3c. Use the .apply method on the url column you created above to retrieve the description text for all of the jobs.